# Phase 3: Churn Prediction API with FastAPI

## Serving the ML Model as a REST API

**What this phase adds to the project:**

| Component | Technology | Purpose |
|---|---|---|
| **API Framework** | FastAPI | Serve model predictions via HTTP endpoints |
| **Data Validation** | Pydantic | Type-safe request/response schemas |
| **API Documentation** | Swagger / OpenAPI | Auto-generated interactive docs |
| **Testing** | pytest + TestClient | Automated endpoint testing |
| **Containerisation** | Docker | Package the API for deployment |

**Why this matters for hiring managers:**
Most data science candidates can train a model in a notebook. Very few can **serve** that model as a production-ready API. This phase shows you can build the bridge between "model in a notebook" and "model in production" — a skill that's increasingly expected in Data Scientist and ML Engineer roles.

**Endpoints built:**

| Method | Endpoint | Description |
|---|---|---|
| `GET` | `/health` | Service status and model info |
| `POST` | `/predict` | Single customer churn prediction |
| `POST` | `/predict/batch` | Batch predictions (up to 1000 customers) |

---

## 0. Environment Setup (Google Colab)

In [17]:
# ── Google Colab / Drive Setup ──
import sys
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path — UPDATE THIS if your folder structure differs
PROJECT_PATH = "/content/drive/MyDrive/Github/Projects/saas-churn-prediction"

# Change to project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Project files: {os.listdir('.')[:10]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Working directory: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
✓ Project files: ['requirements.txt', 'docker-compose.yml', 'features.csv', '.gitignore', 'Makefile', 'files.zip', 'api', 'data', 'notebooks', 'models']


In [18]:
import sys

if "api.main" in sys.modules:
    del sys.modules["api.main"]

print("✓ Cleared cached api.main")

✓ Cleared cached api.main


In [19]:
import api.main

print(api.main.__file__)

/content/drive/MyDrive/Github/Projects/saas-churn-prediction/api/main.py


In [20]:
# Install dependencies
!pip install fastapi uvicorn pydantic httpx pandas scikit-learn xgboost joblib -q

# Verify model artifacts exist (from Phase 2)
from pathlib import Path

models_dir = Path("models")
required = ["best_model.joblib", "model_metadata.joblib"]
for f in required:
    path = models_dir / f
    if path.exists():
        print(f"  ✓ {f} found")
    else:
        print(f"  ✗ {f} MISSING — run Phase 2 notebook first!")

print("\n✓ Setup complete")

  ✓ best_model.joblib found
  ✓ model_metadata.joblib found

✓ Setup complete


---
## 1. Understanding the API Architecture

Before we run the API, let's understand what each file does and why it's structured this way.

### API File Structure

```
api/
├── __init__.py        ← Makes api/ a Python package
├── main.py            ← FastAPI application (endpoints, model loading, prediction logic)
├── schemas.py         ← Pydantic models (request/response data contracts)
├── Dockerfile         ← Container definition for deployment
└── requirements.txt   ← API-specific dependencies
```

### How FastAPI Works

FastAPI is a modern Python web framework designed specifically for building APIs. It was built on top of two key technologies:

1. **Starlette** — For the web server and routing
2. **Pydantic** — For data validation and serialisation

When a request comes in:
1. FastAPI **validates** the request body against the Pydantic schema
2. If validation passes, it calls the **endpoint function**
3. The function runs the model and returns a response
4. FastAPI **serialises** the response using the response schema

If the request data is invalid (wrong types, missing fields, out-of-range values), FastAPI automatically returns a clear error message — no manual validation code needed.

## 2. Exploring the API Schemas

The schemas define the **data contracts** — what the API expects to receive and what it promises to return. Let's look at them.

In [21]:
# Load and inspect the schemas
from api.schemas import CustomerFeatures, PredictionResponse, HealthResponse, BatchPredictionRequest

# Show what fields a CustomerFeatures request expects
print("═" * 60)
print("  CUSTOMER FEATURES — INPUT SCHEMA")
print("═" * 60)
schema = CustomerFeatures.model_json_schema()
for field_name, field_info in schema.get("properties", {}).items():
    desc = field_info.get("description", "")
    ftype = field_info.get("type", "")
    default = field_info.get("default", "REQUIRED")
    print(f"  {field_name:30s} | {ftype:8s} | default={default}")

print(f"\nTotal fields: {len(schema.get('properties', {}))}")

════════════════════════════════════════════════════════════
  CUSTOMER FEATURES — INPUT SCHEMA
════════════════════════════════════════════════════════════
  monthly_revenue                | number   | default=REQUIRED
  tenure_months                  | integer  | default=REQUIRED
  contract_type                  | integer  | default=REQUIRED
  total_charges                  | number   | default=REQUIRED
  payment_method                 | integer  | default=0
  paperless_billing              | integer  | default=1
  login_frequency                | number   | default=5.0
  feature_adoption_rate          | number   | default=0.5
  days_since_last_login          | integer  | default=3
  avg_session_duration           | number   | default=15.0
  engagement_score               | number   | default=50.0
  support_tickets_total          | integer  | default=2
  support_tickets_last_90d       | integer  | default=0
  escalation_count               | integer  | default=0
  has_phone_service  

In [22]:
# Show the response schema
print("═" * 60)
print("  PREDICTION RESPONSE — OUTPUT SCHEMA")
print("═" * 60)
resp_schema = PredictionResponse.model_json_schema()
for field_name, field_info in resp_schema.get("properties", {}).items():
    desc = field_info.get("description", "no description")
    print(f"  {field_name:30s} → {desc}")

════════════════════════════════════════════════════════════
  PREDICTION RESPONSE — OUTPUT SCHEMA
════════════════════════════════════════════════════════════
  churn_probability              → Probability of churning (0-1)
  churn_prediction               → Whether the customer is predicted to churn at the optimal threshold
  risk_level                     → Risk category: critical, high, medium, or low
  monthly_revenue_at_risk        → Monthly revenue that would be lost if this customer churns (£)
  top_risk_factors               → Top features contributing to the churn prediction


## 3. Starting the API Server

In Colab, we can't run a persistent server the normal way (`uvicorn api.main:app`). Instead, we'll use two approaches:

1. **TestClient** — FastAPI's built-in test client that simulates HTTP requests without a running server (great for development and testing)
2. **Background thread** — Actually start the server in the background so we can hit it with real HTTP requests

Let's start with the TestClient approach, which is how you'd write automated tests.

In [23]:
# Triggers startup explicitly
from fastapi.testclient import TestClient

with TestClient(api.main.app) as client:
    response = client.get("/health")
    print(response.json())


LOADING MODEL ARTIFACTS
PROJECT_ROOT: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
MODELS_DIR: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models

Model path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/best_model.joblib
Exists: True

Metadata path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/model_metadata.joblib
Exists: True

Scaler path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/scaler.joblib
Exists: True

✓ Scaler loaded

✓ Model loaded: logistic_regression
✓ Threshold: 0.7602
✓ Features expected: 45

{'status': 'healthy', 'model_loaded': True, 'model_name': 'logistic_regression', 'optimal_threshold': 0.760233456576927, 'version': '1.0.0'}
Shutting down API...


## 4. Health Check Endpoint

Every production API needs a health check. Load balancers, monitoring tools, and container orchestrators use this to know if the service is alive and ready to serve requests.

**Endpoint:** `GET /health`

In [24]:
# ── Call the health check endpoint ──
with TestClient(api.main.app) as client:
    response = client.get("/health")

print(f"Status code: {response.status_code}")
print(f"\nResponse:")

import json
data = response.json()
for key, value in data.items():
    print(f"  {key:25s}: {value}")


LOADING MODEL ARTIFACTS
PROJECT_ROOT: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
MODELS_DIR: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models

Model path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/best_model.joblib
Exists: True

Metadata path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/model_metadata.joblib
Exists: True

Scaler path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/scaler.joblib
Exists: True

✓ Scaler loaded

✓ Model loaded: logistic_regression
✓ Threshold: 0.7602
✓ Features expected: 45

Shutting down API...
Status code: 200

Response:
  status                   : healthy
  model_loaded             : True
  model_name               : logistic_regression
  optimal_threshold        : 0.760233456576927
  version                  : 1.0.0


## 5. Single Customer Prediction

This is the core endpoint — submit a customer's features and get back a churn risk assessment.

**Endpoint:** `POST /predict`

Let's test it with two very different customer profiles to see how the model responds.

In [25]:
print(response.status_code)
print(response.json())

200
{'status': 'healthy', 'model_loaded': True, 'model_name': 'logistic_regression', 'optimal_threshold': 0.760233456576927, 'version': '1.0.0'}


In [26]:
# ── Test with a loyal, engaged customer ──
loyal_customer = {
    "monthly_revenue": 79.85,
    "tenure_months": 48,
    "contract_type": 2,           # two-year contract
    "total_charges": 3832.80,
    "payment_method": 3,          # credit card (auto-pay)
    "paperless_billing": 1,
    "login_frequency": 12.0,      # logs in frequently
    "feature_adoption_rate": 0.85, # uses most features
    "days_since_last_login": 1,
    "avg_session_duration": 25.0,
    "engagement_score": 82.0,     # highly engaged
    "support_tickets_total": 4,
    "support_tickets_last_90d": 0, # no recent issues
    "escalation_count": 0,
    "has_phone_service": 1,
    "has_internet_service": 1,
    "has_online_security": 1,
    "has_online_backup": 1,
    "has_device_protection": 1,
    "has_tech_support": 1,
    "has_streaming_tv": 1,
    "has_streaming_movies": 1,
    "revenue_per_tenure": 1.66,
    "engagement_to_tenure_ratio": 1.71,
    "support_to_usage_ratio": 0.33,
    "is_senior_citizen": 0,
    "has_partner": 1,
    "has_dependents": 1,
    "gender_encoded": 0,
}

with TestClient(api.main.app) as client:
    response = client.post("/predict", json=loyal_customer)
    result = response.json()


print("═" * 60)
print("  PREDICTION: LOYAL CUSTOMER (48 months, engaged)")
print("═" * 60)
print(f"  Churn probability:     {result['churn_probability']:.1%}")
print(f"  Prediction:            {'WILL CHURN' if result['churn_prediction'] else 'WILL STAY'}")
print(f"  Risk level:            {result['risk_level'].upper()}")
print(f"  Revenue at risk:       £{result['monthly_revenue_at_risk']:.2f}/month")
print(f"  Top risk factors:      {', '.join(result['top_risk_factors'][:3])}")


LOADING MODEL ARTIFACTS
PROJECT_ROOT: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
MODELS_DIR: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models

Model path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/best_model.joblib
Exists: True

Metadata path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/model_metadata.joblib
Exists: True

Scaler path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/scaler.joblib
Exists: True

✓ Scaler loaded

✓ Model loaded: logistic_regression
✓ Threshold: 0.7602
✓ Features expected: 45

Shutting down API...
════════════════════════════════════════════════════════════
  PREDICTION: LOYAL CUSTOMER (48 months, engaged)
════════════════════════════════════════════════════════════
  Churn probability:     100.0%
  Prediction:            WILL CHURN
  Risk level:            CRITICAL
  Revenue at risk:       £79.85/month
  Top risk factors:      monthly_revenue, has_de

In [27]:
# ── Test with a high-risk customer ──
at_risk_customer = {
    "monthly_revenue": 95.00,
    "tenure_months": 2,           # very new
    "contract_type": 0,           # month-to-month (no commitment)
    "total_charges": 190.00,
    "payment_method": 0,          # electronic check (highest churn segment)
    "paperless_billing": 1,
    "login_frequency": 0.5,       # barely logs in
    "feature_adoption_rate": 0.08, # uses almost nothing
    "days_since_last_login": 42,  # hasn't logged in for 6 weeks
    "avg_session_duration": 2.0,
    "engagement_score": 5.0,      # basically disengaged
    "support_tickets_total": 14,  # many complaints
    "support_tickets_last_90d": 9,
    "escalation_count": 4,        # multiple escalations
    "has_phone_service": 1,
    "has_internet_service": 1,
    "has_online_security": 0,
    "has_online_backup": 0,
    "has_device_protection": 0,
    "has_tech_support": 0,
    "has_streaming_tv": 0,
    "has_streaming_movies": 0,
    "revenue_per_tenure": 47.50,
    "engagement_to_tenure_ratio": 2.50,
    "support_to_usage_ratio": 28.0,
    "is_senior_citizen": 0,
    "has_partner": 0,
    "has_dependents": 0,
    "gender_encoded": 1,
}

response = client.post("/predict", json=at_risk_customer)
result = response.json()

print("═" * 60)
print("  PREDICTION: AT-RISK CUSTOMER (2 months, disengaged)")
print("═" * 60)
print(f"  Churn probability:     {result['churn_probability']:.1%}")
print(f"  Prediction:            {'WILL CHURN' if result['churn_prediction'] else 'WILL STAY'}")
print(f"  Risk level:            {result['risk_level'].upper()}")
print(f"  Revenue at risk:       £{result['monthly_revenue_at_risk']:.2f}/month")
print(f"  Top risk factors:      {', '.join(result['top_risk_factors'][:3])}")

════════════════════════════════════════════════════════════
  PREDICTION: AT-RISK CUSTOMER (2 months, disengaged)
════════════════════════════════════════════════════════════
  Churn probability:     100.0%
  Prediction:            WILL CHURN
  Risk level:            CRITICAL
  Revenue at risk:       £95.00/month
  Top risk factors:      monthly_revenue, paperless_billing, has_partner


## 6. Batch Prediction

In production, you'd typically score the entire customer base overnight. The batch endpoint handles up to 1000 customers per request and returns aggregate statistics alongside individual predictions.

**Endpoint:** `POST /predict/batch`

In [28]:
# ── Score multiple customers at once ──
from fastapi.testclient import TestClient
import api.main

# ── Score multiple customers at once ──
import pandas as pd

# ── Score multiple customers at once ──
import numpy as np

# Create realistic batch customers matching the API schema
batch_customers = []

for i in range(20):
    customer = {
        "monthly_revenue": float(np.random.uniform(20, 120)),
        "tenure_months": int(np.random.randint(1, 72)),
        "contract_type": int(np.random.choice([0, 1, 2])),
        "total_charges": float(np.random.uniform(100, 8000)),
        "payment_method": int(np.random.choice([0, 1, 2, 3])),
        "paperless_billing": int(np.random.choice([0, 1])),
        "login_frequency": float(np.random.uniform(0, 15)),
        "feature_adoption_rate": float(np.random.uniform(0, 1)),
        "days_since_last_login": int(np.random.randint(0, 60)),
        "avg_session_duration": float(np.random.uniform(1, 40)),
        "engagement_score": float(np.random.uniform(0, 100)),
        "support_tickets_total": int(np.random.randint(0, 15)),
        "support_tickets_last_90d": int(np.random.randint(0, 10)),
        "escalation_count": int(np.random.randint(0, 5)),
        "has_phone_service": int(np.random.choice([0, 1])),
        "has_internet_service": int(np.random.choice([0, 1])),
        "has_online_security": int(np.random.choice([0, 1])),
        "has_online_backup": int(np.random.choice([0, 1])),
        "has_device_protection": int(np.random.choice([0, 1])),
        "has_tech_support": int(np.random.choice([0, 1])),
        "has_streaming_tv": int(np.random.choice([0, 1])),
        "has_streaming_movies": int(np.random.choice([0, 1])),
        "revenue_per_tenure": float(np.random.uniform(0, 50)),
        "engagement_to_tenure_ratio": float(np.random.uniform(0, 10)),
        "support_to_usage_ratio": float(np.random.uniform(0, 30)),
        "is_senior_citizen": int(np.random.choice([0, 1])),
        "has_partner": int(np.random.choice([0, 1])),
        "has_dependents": int(np.random.choice([0, 1])),
        "gender_encoded": int(np.random.choice([0, 1])),
    }

    batch_customers.append(customer)

# Call the batch endpoint
with TestClient(api.main.app) as client:
    response = client.post("/predict/batch", json={"customers": batch_customers})
    batch_result = response.json()

# Debug output
print("STATUS:", response.status_code)
print("RAW RESPONSE:", batch_result)

print("═" * 60)
print("  BATCH PREDICTION RESULTS (20 customers)")
print("═" * 60)

# Summary
summary = batch_result["summary"]

print(f"\n  Total scored:              {summary['total_customers_scored']}")
print(f"  High/Critical risk:        {summary['high_risk_customers']}")
print(f"  Monthly revenue at risk:   £{summary['total_monthly_revenue_at_risk']:,.2f}")
print(f"  Annual revenue at risk:    £{summary['estimated_annual_revenue_at_risk']:,.2f}")

# Show individual predictions
print(f"\n  ── Individual Predictions ──")
print(f"  {'#':>3}  {'Prob':>6}  {'Risk':>10}  {'Revenue':>10}  {'Prediction'}")
print(f"  {'─'*3}  {'─'*6}  {'─'*10}  {'─'*10}  {'─'*12}")

for i, pred in enumerate(batch_result["predictions"]):
    prob = pred["churn_probability"]
    risk = pred["risk_level"]
    rev = pred["monthly_revenue_at_risk"]
    status = "⚠ CHURN" if pred["churn_prediction"] else "  OK"

    print(f"  {i+1:>3}  {prob:>5.1%}  {risk:>10}  £{rev:>8.2f}  {status}")




LOADING MODEL ARTIFACTS
PROJECT_ROOT: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
MODELS_DIR: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models

Model path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/best_model.joblib
Exists: True

Metadata path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/model_metadata.joblib
Exists: True

Scaler path: /content/drive/MyDrive/Github/Projects/saas-churn-prediction/models/scaler.joblib
Exists: True

✓ Scaler loaded

✓ Model loaded: logistic_regression
✓ Threshold: 0.7602
✓ Features expected: 45

Shutting down API...
STATUS: 200
RAW RESPONSE: {'predictions': [{'churn_probability': 0.4982, 'churn_prediction': False, 'risk_level': 'medium', 'monthly_revenue_at_risk': 0.0, 'top_risk_factors': ['monthly_revenue', 'paperless_billing', 'has_partner', 'is_male', 'is_senior']}, {'churn_probability': 1.0, 'churn_prediction': True, 'risk_level': 'critical', 'monthly_revenue_at_ris

## 7. Input Validation Demo

One of FastAPI's strongest features is automatic input validation via Pydantic. If a request contains invalid data, the API returns a clear error message explaining exactly what's wrong — no custom validation code needed.

This is important in production because it prevents garbage-in-garbage-out predictions.

In [29]:
# ── Test 1: Missing required field ──
print("Test 1: Missing 'monthly_revenue' (required field)")
bad_request = {"tenure_months": 12, "contract_type": 1, "total_charges": 500}
response = client.post("/predict", json=bad_request)
print(f"  Status: {response.status_code} (expected 422 = validation error)")
errors = response.json()["detail"]
for err in errors:
    loc = " → ".join(str(x) for x in err["loc"])
    print(f"  Error: {loc}: {err['msg']}")

print()

# ── Test 2: Negative revenue ──
print("Test 2: Negative monthly revenue")
bad_request_2 = {**loyal_customer, "monthly_revenue": -50}
response = client.post("/predict", json=bad_request_2)
print(f"  Status: {response.status_code} (expected 422)")

print()

# ── Test 3: Invalid contract type ──
print("Test 3: Contract type = 99 (must be 0, 1, or 2)")
bad_request_3 = {**loyal_customer, "contract_type": 99}
response = client.post("/predict", json=bad_request_3)
print(f"  Status: {response.status_code} (expected 422)")

print()

# ── Test 4: Empty batch ──
print("Test 4: Empty batch (must have at least 1 customer)")
response = client.post("/predict/batch", json={"customers": []})
print(f"  Status: {response.status_code} (expected 422)")

print()
print("✓ All validation tests passed — the API correctly rejects bad input")

Test 1: Missing 'monthly_revenue' (required field)
  Status: 422 (expected 422 = validation error)
  Error: body → monthly_revenue: Field required

Test 2: Negative monthly revenue
  Status: 422 (expected 422)

Test 3: Contract type = 99 (must be 0, 1, or 2)
  Status: 422 (expected 422)

Test 4: Empty batch (must have at least 1 customer)
  Status: 422 (expected 422)

✓ All validation tests passed — the API correctly rejects bad input


## 8. Running Automated Tests

The `tests/test_api.py` file contains structured pytest tests. Let's run them to verify everything works.

In [30]:
!cd {PROJECT_PATH} && python -m pytest tests/test_api.py -v --tb=short 2>&1

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Github/Projects/saas-churn-prediction
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
collected 18 items                                                             

tests/test_api.py::TestHealthEndpoint::test_health_returns_200 PASSED    [  5%]
tests/test_api.py::TestHealthEndpoint::test_health_response_structure PASSED [ 11%]
tests/test_api.py::TestHealthEndpoint::test_health_model_is_loaded PASSED [ 16%]
tests/test_api.py::TestPredictEndpoint::test_predict_returns_200 PASSED  [ 22%]
tests/test_api.py::TestPredictEndpoint::test_predict_response_structure PASSED [ 27%]
tests/test_api.py::TestPredictEndpoint::test_predict_probability_range PASSED [ 33%]
tests/test_api.py::TestPredictEndpoint::test_predict_risk_level_valid PASSED [ 38%]
tests/test_api.py::TestPre

## 9. Running as a Real Server (Optional)

Below shows how to start the API as a real HTTP server in Colab using a background thread. This lets you test with actual HTTP requests and see the auto-generated Swagger documentation.

 This uses `ngrok` or Colab's port-forwarding to make the server accessible — it's optional and mainly useful for demo purposes.

In [31]:
# ── Start the server in a background thread ──
import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

print("✓ API server running on port 8000")
print("\nIn a local environment, you'd access:")
print("  • Swagger docs:  http://localhost:8000/docs")
print("  • ReDoc:          http://localhost:8000/redoc")
print("  • Health check:   http://localhost:8000/health")

Exception in thread Thread-23 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_25885/3468050722.py", line 7, in run_server
NameError: name 'app' is not defined. Did you mean: 'api'?


✓ API server running on port 8000

In a local environment, you'd access:
  • Swagger docs:  http://localhost:8000/docs
  • ReDoc:          http://localhost:8000/redoc
  • Health check:   http://localhost:8000/health


In [33]:
# Http test cell
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

resp = client.get("/health")
print(resp.status_code, resp.json())

resp = client.post("/predict", json=loyal_customer)
print(resp.status_code, resp.json())

200 {'status': 'healthy', 'model_loaded': True, 'model_name': 'logistic_regression', 'optimal_threshold': 0.760233456576927, 'version': '1.0.0'}
200 {'churn_probability': 1.0, 'churn_prediction': True, 'risk_level': 'critical', 'monthly_revenue_at_risk': 79.85, 'top_risk_factors': ['monthly_revenue', 'has_dependents', 'has_partner', 'paperless_billing', 'is_male']}


---
## Summary — Phase 3 Complete

### What was built

| Component | File | Purpose |
|---|---|---|
| **FastAPI App** | `api/main.py` | 3 endpoints: `/health`, `/predict`, `/predict/batch` |
| **Pydantic Schemas** | `api/schemas.py` | Type-safe request/response contracts with validation |
| **Dockerfile** | `api/Dockerfile` | Container definition for deployment |
| **API Tests** | `tests/test_api.py` | 15+ automated tests covering all endpoints |

### Technologies demonstrated

- **FastAPI** — Modern async API framework (industry standard for ML serving)
- **Pydantic** — Data validation and serialisation
- **Swagger/OpenAPI** — Auto-generated interactive API documentation
- **pytest** — Automated testing with FastAPI TestClient
- **Docker** — Containerised deployment
- **REST API design** — Proper HTTP methods, status codes, error handling


### Running the API locally (outside Colab)

```bash
# From the project root
pip install -r api/requirements.txt
uvicorn api.main:app --reload --port 8000

# Open http://localhost:8000/docs for interactive Swagger UI
```

### Next: Phase 4 — Streamlit Dashboard